In [2]:
import pandas as pd

nome_file = 'children-out-school.csv' 
df = pd.read_csv('../../datasets/raw/children-out-school/children-out-school.csv')

colonne_da_tenere = [
    "iso_code", 
    "region_group", 
    "country", 
    "year", 
    "category", 
    "region", 
    "comp_prim_v2_m", 
    "eduout_prim_m"
]

df_filtrato = df[colonne_da_tenere]
print(df_filtrato.head())

  iso_code               region_group      country  year   category region  \
0      AFG  Central and Southern Asia  Afghanistan  2015  Ethnicity    NaN   
1      AFG  Central and Southern Asia  Afghanistan  2015  Ethnicity    NaN   
2      AFG  Central and Southern Asia  Afghanistan  2015  Ethnicity    NaN   
3      AFG  Central and Southern Asia  Afghanistan  2015  Ethnicity    NaN   
4      AFG  Central and Southern Asia  Afghanistan  2015  Ethnicity    NaN   

   comp_prim_v2_m  eduout_prim_m  
0          0.3489            NaN  
1          0.6573            NaN  
2          0.4849            NaN  
3          0.3656            NaN  
4          0.4102            NaN  


/var/folders/nj/kxr7k46n76q3w6dmc0rj_53r0000gn/T/ipykernel_39868/3447323748.py:4: DtypeWarning: Columns (7,12,13,14,15,16,17,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../../datasets/raw/children-out-school/children-out-school.csv')


In [3]:
regioni_target = ['Northern Africa and Western Asia', 'Sub-Saharan Africa']
categoria_target = 'Region'

df_filtrato_africa_asia = df_filtrato[
    (df_filtrato['region_group'].isin(regioni_target)) & 
    (df_filtrato['category'] == categoria_target)
]
print(df_filtrato_africa_asia.head())

#df_filtrato_africa_asia.to_csv('dataset_africa_asia.csv', index=False)

    iso_code        region_group country  year category          region  \
913      AGO  Sub-Saharan Africa  Angola  2015   Region           Bengo   
914      AGO  Sub-Saharan Africa  Angola  2015   Region        Benguela   
915      AGO  Sub-Saharan Africa  Angola  2015   Region             Bie   
916      AGO  Sub-Saharan Africa  Angola  2015   Region         Cabinda   
917      AGO  Sub-Saharan Africa  Angola  2015   Region  Cuando Cubango   

     comp_prim_v2_m  eduout_prim_m  
913          0.5377         0.3201  
914          0.6168         0.2069  
915          0.4104         0.3247  
916          0.6905         0.1712  
917          0.2633         0.4733  


In [4]:
df_continenti = pd.read_csv('../../datasets/raw/country-codes/country-code.csv')
df_continenti.drop_duplicates(subset=['Three_Letter_Country_Code'], inplace=True)

# left_on e right_on si usano quando le colonne chiave hanno nomi diversi
df_unito = pd.merge(
    df_filtrato_africa_asia, 
    df_continenti[['Three_Letter_Country_Code', 'Continent_Name']], 
    left_on='iso_code', 
    right_on='Three_Letter_Country_Code', 
    how='left'
)

df_africa = df_unito[df_unito['Continent_Name'] == 'Africa'].copy()

df_africa.drop(columns=['region_group','category','Continent_Name','Three_Letter_Country_Code'], inplace=True)

# pulizia caratteri speciali
df_africa['country'] = df_africa['country'].str.replace("CÃÂ´te d'Ivoire", "Côte d'Ivoire", regex=False)
df_africa = df_africa[~df_africa['region'].str.contains('RÃÂ©Gion', na=False)]
df_africa = df_africa[~df_africa['region'].str.contains('<', na=False)]
df_africa['region'] = df_africa['region'].str.replace(r'^\]', '', regex=True)

print(df_africa.head())

  iso_code country  year          region  comp_prim_v2_m  eduout_prim_m
0      AGO  Angola  2015           Bengo          0.5377         0.3201
1      AGO  Angola  2015        Benguela          0.6168         0.2069
2      AGO  Angola  2015             Bie          0.4104         0.3247
3      AGO  Angola  2015         Cabinda          0.6905         0.1712
4      AGO  Angola  2015  Cuando Cubango          0.2633         0.4733


In [6]:
#creo un dataset per la percentuale di bambini fuori scuola in Africa
df_africa_copy = df_africa.copy()

df_africa_copy.drop(columns=['comp_prim_v2_m'], inplace=True)
df_africa_copy.dropna(subset=['eduout_prim_m'], inplace=True)
edu_out_grouped = (
    df_africa_copy.groupby(['iso_code', 'country', 'region'], as_index=False)
    .agg(
        edu_out_avg=('eduout_prim_m', 'mean'),
    )
    .round(4)
)

edu_out_grouped.to_csv('out-of-school-grouped.csv', index=False)

In [7]:
#creo un dataset per la percentuale di bambini che hanno completato la scuola primaria in Africa
df_africa_copy2 = df_africa.copy()

df_africa_copy2.drop(columns=['eduout_prim_m'], inplace=True)
df_africa_copy2.dropna(subset=['comp_prim_v2_m'], inplace=True)
comp_primary_grouped = (
    df_africa_copy2.groupby(['iso_code', 'country', 'region'], as_index=False)
    .agg(
        comp_prim_avg=('comp_prim_v2_m', 'mean'),
    )
    .round(4)
)

comp_primary_grouped.to_csv('comp-primary-grouped.csv', index=False)